# Checking the data files

This notebook checks which files we have and what is stored in them.

It opens one ROOT file from each folder and lists the variables inside. It reads information about the files without moving or removing any data.

In [ ]:
import json
import platform
import sys
import importlib.util
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

HERE = Path.cwd().resolve()
CANDIDATES = list(dict.fromkeys([HERE, *HERE.parents, Path.home() / "Desktop" / "PPP"]))
MANIFEST_NAME = "DataForEmre_verified_manifest_20261029.json"
PROJECT = next((p for p in CANDIDATES
                if (p / "analysis" / "data_paths.py").is_file()), None)

if PROJECT is not None:
    MODE = "project"
    sys.path.insert(0, str(PROJECT / "analysis"))
    from data_paths import resolve_data_paths
    SOURCE, MANIFEST = resolve_data_paths(PROJECT)
    OUTPUT = PROJECT / "analysis" / "outputs" / "data_inventory" / "schema_inventory.json"
else:
    PROJECT = next((p for p in CANDIDATES
                    if (p / "DataForEmre_transfer_20261029" / "verified_manifest.json").is_file()), None)
    if PROJECT is None:
        raise FileNotFoundError(
            "Cannot find the data and saved file list. Open this notebook from "
            "the repository's analysis folder and choose the 'PPP first look' Python kernel."
        )
    MODE = "CERN"
    MANIFEST = PROJECT / "DataForEmre_transfer_20261029" / "verified_manifest.json"
    OUTPUT = PROJECT / "outputs" / "data_inventory" / "schema_inventory.json"

manifest = json.loads(MANIFEST.read_text())
if MODE == "CERN":
    SOURCE = Path(manifest["source"])
if not SOURCE.is_dir():
    raise FileNotFoundError(f"Data folder is not accessible: {SOURCE}")
if not any(importlib.util.find_spec(name) is not None for name in ("uproot", "ROOT")):
    raise ModuleNotFoundError(
        "This Python environment has no ROOT reader. For the local notebook, "
        "choose 'PPP first look' or the interpreter analysis/.venv/bin/python."
    )

records = manifest["files"]
for record in records:
    relative = PurePosixPath(record["path"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe file-list path: {relative}")

print("Running on:", MODE)
print("Python:", sys.executable)
print("Data:", SOURCE)
print("Saved file list:", MANIFEST)
print("File list date:", manifest.get("created_utc", "not recorded"))
print("Python version:", sys.version.split()[0], "on", platform.system())
if MODE == "project":
    print("File sizes are compared with the private manifest; modification dates are not checked in project mode.")

## Which files do we have?

The first table shows the number of files and their total size in each folder, using the file list saved during downloading. It is not a new scan of the CERN folder.

Each row counts files directly inside that folder. A parent folder can therefore show zero files even when its subfolders contain data.

For a quick check, the code picks the smallest nonempty ROOT file in each folder. This gives us a first look at the format; the other files could still differ.

A folder name tells us where a sample came from. For example, `unpaired` does not mean that every jet in it is BIB.


In [ ]:
grouped = defaultdict(list)
for record in records:
    grouped[str(PurePosixPath(record["path"]).parent)].append(record)
directories = sorted(set(manifest.get("directories", [])) | set(grouped))
print(f"{'Folder in saved file list':90} {'Files':>7} {'GB (decimal)':>13}")
print("-" * 114)
for directory in directories:
    group = grouped[directory]
    # Parent-folder rows show files directly inside, not recursive totals.
    print(f"{directory:90} {len(group):7d} {sum(r['size'] for r in group)/1e9:13.4f}")
print(f"TOTAL: {len(records)} files, {sum(r['size'] for r in records)/1e9:.4f} GB")
samples = []
for directory, group in sorted(grouped.items()):
    eligible = [r for r in group if r["path"].endswith(".root") and r["size"] > 0]
    if eligible:
        samples.append((directory, min(eligible, key=lambda r: (r["size"], r["path"]))))
print("\nMetadata samples:")
for directory, record in samples:
    print(directory, "->", PurePosixPath(record["path"]).name)


## What is inside the files?

ROOT uses a few terms that appear in the output:

- A **tree** is like a table.
- A **branch** is a stored variable, like a column. It can contain a list of values.
- An **entry** is one row. It is not necessarily one jet or one event.

The code below reads the names and types of the variables, and the number of entries. It does not read the actual event values yet. For the downloaded files, it also compares their sizes with the saved list. Their modification dates changed during downloading, so those dates are not used to flag changed source files. This notebook does not calculate new checksums.

The useful findings from this check are:

- The LLP and unpaired examples contain a `JetContent` table prepared for analysis.
- The ZMUMU example uses ATLAS's AOD format and needs different handling.
- We have variables such as `jet_pt`, `jet_time_ATLAS`, `jet_time_spread`, `cell_z` and `cell_time` to explore next.

The last code cell prints the first 30 variables from each tree. The full list is saved in `outputs/data_inventory/schema_inventory.json`.

The variable names alone do not tell us their units or which cells belong to which jets. We need those definitions from Emil or the code that made the files.


In [ ]:
def inspect_uproot(path):
    import uproot
    result = {"backend": "uproot", "version": uproot.__version__, "objects": []}
    with uproot.open(str(path)) as handle:
        for key, classname in handle.classnames(recursive=True).items():
            item = {"key": key, "class": classname}
            if classname.startswith("TTree") or "RNTuple" in classname:
                try:
                    tree = handle[key]
                    item["entries"] = int(tree.num_entries)
                    item["branches"] = dict(tree.typenames())
                except Exception as exc:
                    item["metadata_error"] = f"{type(exc).__name__}: {exc}"
            result["objects"].append(item)
    return result

def inspect_pyroot(path):
    import ROOT
    result = {"backend": "ROOT", "version": ROOT.gROOT.GetVersion(), "objects": []}
    handle = ROOT.TFile.Open(str(path), "READ")
    if not handle or handle.IsZombie():
        raise OSError("ROOT could not open a healthy file")
    def visit(directory, prefix=""):
        for key in directory.GetListOfKeys():
            name = key.GetName()
            classname = key.GetClassName()
            item = {"key": prefix + name + ";" + str(key.GetCycle()), "class": classname}
            if classname.startswith("TDirectory"):
                result["objects"].append(item)
                visit(directory.Get(name), prefix + name + "/")
                continue
            if classname.startswith("TTree"):
                try:
                    tree = directory.Get(name)
                    item["entries"] = int(tree.GetEntries())
                    item["branches"] = {}
                    def branches(items, parent=""):
                        for branch in items:
                            branch_name = parent + branch.GetName()
                            item["branches"][branch_name] = {
                                "class": branch.GetClassName(),
                                "leaves": {leaf.GetName(): leaf.GetTypeName()
                                           for leaf in branch.GetListOfLeaves()},
                            }
                            if branch.GetListOfBranches().GetEntries():
                                branches(branch.GetListOfBranches(), branch_name + "/")
                    branches(tree.GetListOfBranches())
                except Exception as exc:
                    item["metadata_error"] = f"{type(exc).__name__}: {exc}"
            elif "RNTuple" in classname:
                item["metadata_error"] = "RNTuple key found; this PyROOT fallback only inventories TTree branches"
            result["objects"].append(item)
    try:
        visit(handle)
    finally:
        handle.Close()
    return result


In [ ]:
inventory = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "manifest": str(MANIFEST.relative_to(PROJECT)) if MANIFEST.is_relative_to(PROJECT) else str(MANIFEST),
    "manifest_created_utc": manifest.get("created_utc"),
    "source": str(SOURCE),
    "original_cern_source": manifest.get("source"),
    "mode": MODE,
    "mtime_check": "Compared with source record" if MODE == "CERN" else "Not compared: downloaded copies have local timestamps",
    "scope": "ROOT metadata only; one file per populated source folder; no event decoding",
    "python": sys.version,
    "samples": [],
}
for directory, record in samples:
    path = SOURCE / record["path"]
    sample = {"group": directory, "path": record["path"],
              "recorded_size": record["size"], "recorded_mtime_ns": record.get("mtime_ns"),
              "recorded_sha256": record.get("sha256"), "attempt_errors": []}
    try:
        stat = path.stat()
        sample.update(live_size=stat.st_size, live_mtime_ns=stat.st_mtime_ns,
                      size_matches_snapshot=(stat.st_size == record["size"]))
        if MODE == "CERN" and record.get("mtime_ns") is not None:
            sample["mtime_matches_snapshot"] = stat.st_mtime_ns == record["mtime_ns"]
        for reader in (inspect_uproot, inspect_pyroot):
            try:
                sample["schema"] = reader(path)
                partial = any("metadata_error" in item for item in sample["schema"]["objects"])
                sample["status"] = "partial_metadata" if partial else "ok_metadata"
                break
            except Exception as exc:
                sample["attempt_errors"].append({"reader": reader.__name__,
                    "error": f"{type(exc).__name__}: {exc}"})
        else:
            sample["status"] = "failed"
    except Exception as exc:
        sample["status"] = "failed"
        sample["access_error"] = f"{type(exc).__name__}: {exc}"
    inventory["samples"].append(sample)
    print(sample["status"], directory)
    if "schema" in sample:
        print("  Backend:", sample["schema"]["backend"], sample["schema"]["version"])
        for item in sample["schema"]["objects"]:
            if "entries" in item:
                print(" ", item["key"], ":", item["entries"], "entries;",
                      len(item.get("branches", {})), "branches")
            if "metadata_error" in item:
                print("  Metadata error:", item["metadata_error"])
    if sample["attempt_errors"]:
        print("  Attempt errors:", sample["attempt_errors"])
    if "access_error" in sample:
        print("  Access error:", sample["access_error"])
    if sample.get("size_matches_snapshot") is False or sample.get("mtime_matches_snapshot") is False:
        print("  NOTICE: live file differs from the recorded size/time snapshot")

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\n")
print("\nSaved:", OUTPUT)
print("Metadata successes:", sum(s["status"] == "ok_metadata" for s in inventory["samples"]),
      "/", len(inventory["samples"]))


partial = [s["group"] for s in inventory["samples"] if s["status"] == "partial_metadata"]
failed = [s["group"] for s in inventory["samples"] if s["status"] == "failed"]
size_mismatches = [s["group"] for s in inventory["samples"] if s.get("size_matches_snapshot") is False]
print("Partial reads:", len(partial), "| Failed reads:", len(failed), "| Size mismatches:", len(size_mismatches))
if partial or failed or size_mismatches:
    raise RuntimeError("Some files need attention. See the details above and the saved report.")


In [ ]:
# Keep the meeting preview short; the JSON retains every branch.
PREVIEW_BRANCHES = 60
for sample in inventory["samples"]:
    print("\n", sample["group"])
    for item in sample.get("schema", {}).get("objects", []):
        if "branches" in item:
            print(" ", item["key"])
            branch_items = list(item["branches"].items())
            for name, typename in branch_items[:PREVIEW_BRANCHES]:
                print("   ", name, "->", typename)
            if len(branch_items) > PREVIEW_BRANCHES:
                print("   ...", len(branch_items) - PREVIEW_BRANCHES,
                      "more variables saved in", str(OUTPUT))


## Questions

- What does one entry represent? How do we connect its jets, clusters and cells?
- Which selection identifies BIB in the unpaired data? Which jets in the LLP sample should count as signal?
- What units are used for energy, position and time? Are the times already corrected for travel from the collision point?
- Which trigger and quality cuts have already been applied?
- What ATLAS software or existing code should we use to read the ZMUMU AOD files?
- Which identifiers let us keep jets from the same event together when splitting training and testing? Are there event weights we should use?